# Solutions · 00 · Models of linear systems

Worked solutions to the exercises of [notebook 00](../notebooks/00_models_of_linear_systems.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere englincontrol is missing): install it from GitHub
    import englincontrol
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/englincontrol"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from englincontrol import systems as sy, models, response as rs, statespace as sp
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Derive the state-space model of the two-tank system from mass balances (A h' = q_in - q_out with q = h/R across each valve) and confirm `models.two_tanks()`'s matrices and dc gain R2.

In [2]:
A1, A2, R1, R2 = 1.0, 1.0, 2.0, 3.0
# mass balances: A1 h1' = q - (h1 - h2)/R1 ;  A2 h2' = (h1 - h2)/R1 - h2/R2
A = np.array([[-1 / (A1 * R1), 1 / (A1 * R1)], [1 / (A2 * R1), -1 / (A2 * R1) - 1 / (A2 * R2)]])
B = np.array([[1 / A1], [0.0]]); C = np.array([[0.0, 1.0]])
m = models.two_tanks(A1, A2, R1, R2)
print("A matches the library:", np.allclose(A, m["system"].A), "; B:", np.allclose(B, m["system"].B))
print(f"dc gain of level 2: {sy.ss(A, B, C, 0.0).dcgain()[0, 0]:.4f} = R2 (at steady state the whole inflow leaves through R2, so h2 = q R2)")

A matches the library: True ; B: True
dc gain of level 2: 3.0000 = R2 (at steady state the whole inflow leaves through R2, so h2 = q R2)


Each tank stores volume at the rate of its net flow; the flows through the valves are proportional to the
head across them. The resulting matrices are the library's, and the dc gain R₂ has a physical reading: at
steady state all the inflow passes through the outlet valve, so the second level is q R₂ whatever the first
valve does.

## Exercise 2

Realise (s + 3)/((s + 1)(s + 2)(s + 3)) in controllable canonical form; is it minimal? Use `systems.minimal` and explain which mode is lost and why it is harmless here.

In [3]:
G = sy.tf(np.polymul([1.0, 3.0], [1.0]), np.polymul(np.polymul([1.0, 1.0], [1.0, 2.0]), [1.0, 3.0]))
print("transfer function as written (no cancellation performed):", np.round(G.num, 3), "/", np.round(G.den, 3))
S_raw = sy.StateSpace(np.array([[-6.0, -11.0, -6.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]]), np.array([[1.0], [0.0], [0.0]]), np.array([[0.0, 1.0, 3.0]]), np.array([[0.0]]))
print(f"controllable canonical realisation: {S_raw.n} states; controllable: {sp.controllability(S_raw.A, S_raw.B)['controllable']}, observable: {sp.observability(S_raw.A, S_raw.C)['observable']}")
print(f"minimal realisation: {sy.minimal(S_raw).n} states:", sy.minimal(S_raw).to_tf())

transfer function as written (no cancellation performed): [1. 3.] / [ 1.  6. 11.  6.]
controllable canonical realisation: 3 states; controllable: True, observable: False
minimal realisation: 2 states: TransferFunction(1 / s^2 + 3s + 2)


The controllable canonical form is controllable by construction, but the mode at −3 is unobservable: the
numerator's zero at −3 cancels it in the output. The minimal realisation has two states and the transfer
function 1/((s+1)(s+2)). Losing a stable mode is harmless to the input-output behaviour (it decays on its own);
losing an unstable one would hide a runaway, as notebook 08 shows.

## Exercise 3

**Implement it yourself:** write the series connection of two state-space systems as block matrices (the output of the first feeds the second) and check your result against `systems.series` on the DC motor followed by a first-order sensor.

In [4]:
motor = models.dc_motor()["system"]
sensor = sy.ss([[-50.0]], [[50.0]], [[1.0]], 0.0)             # first-order sensor with a 20 ms time constant
# series: output of the motor feeds the sensor
A = np.block([[motor.A, np.zeros((2, 1))], [sensor.B @ motor.C, sensor.A]])
B = np.vstack([motor.B, sensor.B @ motor.D]); C = np.hstack([sensor.D @ motor.C, sensor.C]); D = sensor.D @ motor.D
hand = sy.ss(A, B, C, D); lib = sy.series(motor, sensor)
print("poles by hand:", np.round(np.sort(hand.poles().real), 4), " library:", np.round(np.sort(lib.poles().real), 4))
print("transfer functions equal:", np.allclose(hand.to_tf().den, lib.to_tf().den) and np.allclose(hand.to_tf().num, lib.to_tf().num))

poles by hand: [-50.      -9.9975  -2.0025]  library: [-50.      -9.9975  -2.0025]
transfer functions equal: True


Stacking the states, the second system's input is the first's output: its B multiplies the first's C in the
lower-left block. The poles of the cascade are the union of the two sets, and the transfer function is the
product - the library's `series` does exactly this.